# AI4I 2020 Predictive Maintenance: Which machines should be inspected first?
# AI4I 2020 預測維護：哪些機器最需要優先檢查？

**Flow / 流程:** load data → explore → model (avoiding leakage) → compare with rule-based → risk ranking → database → business value
載入資料 → 探索 → 建模（避開資料洩漏）→ 與規則式比較 → 風險排序 → 資料庫 → 商業價值

**How to use / 使用方式:** Run top to bottom in Colab. The first cell asks you to upload `ai4i2020.csv`.
在 Colab 由上往下依序執行，第一格會要求上傳 `ai4i2020.csv`。

> Chart labels are in English because Colab's default chart font cannot show Chinese.
> 圖表標籤使用英文，因為 Colab 預設的圖表字型無法顯示中文。

In [ ]:
import io
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

up = files.upload()  # choose ai4i2020.csv / 選擇 ai4i2020.csv
df = pd.read_csv(io.BytesIO(next(iter(up.values()))))
df.columns = ["udi", "product_id", "type", "air_temp", "process_temp", "rpm",
              "torque", "tool_wear", "failure", "twf", "hdf", "pwf", "osf", "rnf"]
print(df.shape)
df.head()

## 1. Data check and exploration / 資料檢查與探索

In [ ]:
print("Missing values / 缺值:", df.isna().sum().sum())
print("Failure rate / 故障率:", round(df["failure"].mean() * 100, 2), "%")
print(df.groupby("type")["failure"].agg(["count", "mean"]))
df[["twf", "hdf", "pwf", "osf", "rnf"]].sum()

## 2. Features / 特徵

Two derived features are added: temperature difference and power.
加入兩個衍生特徵：溫差與功率。

**Data leakage warning / 資料洩漏提醒:** `failure` is derived from the five failure-type columns `twf/hdf/pwf/osf/rnf`, so these must **not** be used as features.
`failure` 是由這五個故障類型欄位推出來的，所以它們**不能**當特徵。

In [ ]:
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power_w"] = df["torque"] * df["rpm"] * 2 * np.pi / 60
df["wear_torque"] = df["tool_wear"] * df["torque"]

FEATURES = ["air_temp", "process_temp", "rpm", "torque", "tool_wear",
            "temp_diff", "power_w", "wear_torque", "type_L", "type_M"]
X = pd.get_dummies(df, columns=["type"], dtype=int)[FEATURES]
y = df["failure"]
X.describe().T[["mean", "min", "max"]]

## 3. Split and model / 切分資料與建模

Stratified split keeps the failure rate the same in train and test. Metrics: PR-AUC, precision, recall, F1 (not accuracy).
分層切分讓訓練與測試集的故障率一致。評估用 PR-AUC、precision、recall、F1，不看 accuracy。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                           min_samples_leaf=3, random_state=42, n_jobs=-1),
}
scores = {}
rows = []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    scores[name] = p
    pred = (p >= 0.5).astype(int)
    rows.append([name, average_precision_score(y_te, p), precision_score(y_te, pred),
                 recall_score(y_te, pred), f1_score(y_te, pred)])
pd.DataFrame(rows, columns=["model", "PR-AUC", "precision", "recall", "F1"]).round(3)

## 4. Rule-based baseline / 規則式基準線

Uses the failure conditions described in the dataset documentation as the "traditional rule". The data is synthetic and failures were generated from these rules, so the rule-based method will be very strong. **Say this honestly in the report.**
用資料集文件描述的故障條件當作「傳統規則」。這是合成資料，故障本來就由這些規則產生，所以規則式會很強。**報告中要誠實說明。**

In [ ]:
def rule_flag(d):
    limit = d["type"].map({"L": 11000, "M": 12000, "H": 13000})
    return (
        ((d["temp_diff"] < 8.6) & (d["rpm"] < 1380))      # heat dissipation / 散熱失效
        | (d["power_w"] < 3500) | (d["power_w"] > 9000)   # power / 電力失效
        | (d["wear_torque"] > limit)                       # overstrain / 過載失效
        | (d["tool_wear"] >= 200)                          # tool wear / 工具磨損
    ).astype(int)

rule_te = rule_flag(df.loc[X_te.index])
print("Rule-based  precision:", round(precision_score(y_te, rule_te), 3),
      " recall:", round(recall_score(y_te, rule_te), 3),
      " share inspected / 檢查比例:", round(rule_te.mean(), 3))

## 5. Maintenance priority: cumulative capture curve / 維護優先順序：累積捕捉曲線

Rank machines from highest to lowest risk. If we inspect only the top k%, what share of real failures do we catch?
把機器依風險由高到低排序。只檢查前 k% 時，能抓到多少比例的真實故障？

In [ ]:
def capture_curve(score, y_true):
    order = np.argsort(-np.asarray(score))
    caught = np.cumsum(np.asarray(y_true)[order]) / y_true.sum()
    return np.arange(1, len(score) + 1) / len(score), caught

plt.figure(figsize=(7, 5))
for name, p in scores.items():
    xs, ys = capture_curve(p, y_te)
    plt.plot(xs, ys, label=name)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random inspection")
plt.scatter([rule_te.mean()], [recall_score(y_te, rule_te)], color="red", zorder=5, label="Rule-based")
plt.xlabel("Share of machines inspected"); plt.ylabel("Share of failures caught")
plt.title("Cumulative capture curve"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 6. Store in a database (SQLite) / 存入資料庫（SQLite）

SQLite is built into Colab, nothing to install. **If your course requires another database, rewrite this section for it.**
SQLite 是 Colab 內建的，不用安裝。**若課程指定其他資料庫，這一段要改成該資料庫的寫法。**

In [ ]:
con = sqlite3.connect("ai4i.db")

machines = df[["product_id", "type"]].drop_duplicates()
readings = df[["udi", "product_id", "air_temp", "process_temp", "rpm", "torque", "tool_wear"]]
failures = df[["udi", "failure", "twf", "hdf", "pwf", "osf", "rnf"]]
machines.to_sql("machines", con, if_exists="replace", index=False)
readings.to_sql("sensor_readings", con, if_exists="replace", index=False)
failures.to_sql("failures", con, if_exists="replace", index=False)

best = "RandomForest"
pred = pd.DataFrame({"udi": df.loc[X_te.index, "udi"].values, "risk_score": scores[best]})
pred["rank"] = pred["risk_score"].rank(ascending=False, method="first").astype(int)
pred.to_sql("predictions", con, if_exists="replace", index=False)

pd.read_sql("""
SELECT p.rank, p.udi, m.product_id, m.type, ROUND(p.risk_score, 3) AS risk,
       s.tool_wear, s.rpm, s.torque, f.failure AS actual_failure
FROM predictions p
JOIN sensor_readings s ON s.udi = p.udi
JOIN machines m ON m.product_id = s.product_id
JOIN failures f ON f.udi = p.udi
ORDER BY p.rank LIMIT 20
""", con)

## 7. Business value / 商業價值

Costs are **assumptions**. List them in the report and run a sensitivity analysis (section 12).
成本為**假設值**，需在報告中列明，並做敏感度分析（第 12 節）。

In [ ]:
COST_INSPECT = 50      # cost per inspection (assumption) / 每次檢查成本（假設）
COST_MISS = 1000       # loss per missed failure (assumption) / 漏掉一次故障的損失（假設）

def total_cost(flag, y_true):
    flag = np.asarray(flag); y_true = np.asarray(y_true)
    return flag.sum() * COST_INSPECT + ((flag == 0) & (y_true == 1)).sum() * COST_MISS

n = len(y_te)
rf = scores["RandomForest"]
res = {
    "Inspect none": total_cost(np.zeros(n), y_te),
    "Inspect all": total_cost(np.ones(n), y_te),
    "Rule-based": total_cost(rule_te, y_te),
    "RandomForest (0.5)": total_cost(rf >= 0.5, y_te),
}
# cost-based threshold: inspect when risk > inspection cost / miss cost
# 依成本換算的門檻：風險機率 > 檢查成本 / 漏檢損失 就檢查
t = COST_INSPECT / COST_MISS
res[f"RandomForest (threshold {t:.2f})"] = total_cost(rf >= t, y_te)
pd.Series(res, name="total cost / 總成本").round(0)

## 8. Stricter evaluation: cross-validation and leakage demo / 更嚴謹的評估：交叉驗證與資料洩漏對比

One split can be lucky or unlucky, so use 5-fold cross-validation. We also show that adding the five failure-type columns makes the score look falsely good.
只切一次資料結果可能碰巧好或差，所以用 5 折交叉驗證。同時展示：若把五個故障類型欄位加進特徵，分數會假好。

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                               min_samples_leaf=3, random_state=42, n_jobs=-1)

cv_pr = cross_val_score(rf_cv, X, y, cv=cv, scoring="average_precision")
print("RandomForest 5-fold PR-AUC:", cv_pr.round(3), " mean / 平均:", round(cv_pr.mean(), 3))

# Leakage demo: do NOT use this result, it only illustrates the problem
# 資料洩漏對比：這個結果不能用，只用來說明問題
X_leak = pd.concat([X, df[["twf", "hdf", "pwf", "osf", "rnf"]]], axis=1)
leak_pr = cross_val_score(rf_cv, X_leak, y, cv=cv, scoring="average_precision")
print("With failure-type columns / 含故障類型欄位:", round(leak_pr.mean(), 3),
      "(falsely high: the answer is inside the features / 假好，答案已在特徵裡)")

## 9. Probability calibration / 機率校正

`class_weight` inflates probabilities, so "risk 0.3" does not mean a real 30% chance. Cost calculations need trustworthy probabilities, so we use an unweighted model and calibrate it.
`class_weight` 會讓機率偏高，「風險 0.3」不代表真的有 30% 會故障。成本計算需要可信的機率，所以改用不加權的模型再做校正。

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

rf_plain = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)
cal = CalibratedClassifierCV(rf_plain, method="isotonic", cv=5).fit(X_tr, y_tr)
p_cal = cal.predict_proba(X_te)[:, 1]
p_raw = scores["RandomForest"]

print("Brier score (lower is better / 越小越好)  weighted:", round(brier_score_loss(y_te, p_raw), 4),
      " calibrated / 校正後:", round(brier_score_loss(y_te, p_cal), 4))

plt.figure(figsize=(5, 5))
for label, p in [("Weighted (raw)", p_raw), ("Calibrated", p_cal)]:
    frac, mean_p = calibration_curve(y_te, p, n_bins=8, strategy="quantile")
    plt.plot(mean_p, frac, marker="o", label=label)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("Predicted probability"); plt.ylabel("Actual failure share")
plt.legend(); plt.title("Calibration curve"); plt.show()

## 10. Hybrid: use the rule result as a feature / 混合方法：把規則式結果當作特徵給模型

In [ ]:
X_h = X.assign(rule=rule_flag(df))
rf_h = RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                              min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_h.fit(X_h.loc[X_tr.index], y_tr)
p_h = rf_h.predict_proba(X_h.loc[X_te.index])[:, 1]

print("PR-AUC  ML only / 純 ML:", round(average_precision_score(y_te, scores["RandomForest"]), 3),
      " hybrid / 混合:", round(average_precision_score(y_te, p_h), 3))

## 11. SHAP: why are these machines ranked first? / 用 SHAP 解釋：為什麼這些機器排前面

In [ ]:
!pip install shap -q
import shap

rf_model = models["RandomForest"]
sample = X_te.sample(500, random_state=42)
sv = shap.TreeExplainer(rf_model)(sample)
vals = sv[..., 1] if sv.values.ndim == 3 else sv   # failure class / 取「故障」這一類
shap.plots.beeswarm(vals)

## 12. Cost sensitivity analysis / 成本敏感度分析

The cost of a missed failure is the most uncertain assumption, so we try several values and check whether the conclusion changes.
漏檢損失是最不確定的假設，所以設成多種情況，看結論會不會改變。

In [ ]:
COST_INSPECT = 50
rows = []
for ratio in [5, 10, 20, 50]:
    cost_miss = COST_INSPECT * ratio
    def cost(flag):
        flag = np.asarray(flag)
        return flag.sum() * COST_INSPECT + ((flag == 0) & (np.asarray(y_te) == 1)).sum() * cost_miss
    t = COST_INSPECT / cost_miss      # inspect when risk exceeds this / 風險超過這個值就檢查
    rows.append({
        "miss/inspect cost ratio": ratio,
        "Inspect none": cost(np.zeros(len(y_te))),
        "Inspect all": cost(np.ones(len(y_te))),
        "Rule-based": cost(rule_te),
        "ML (calibrated + best threshold)": cost(p_cal >= t),
    })
pd.DataFrame(rows).set_index("miss/inspect cost ratio")

## 13. Priority = risk × impact (impact is a **simulated assumption**) / 優先順序 = 風險 × 影響（影響為**模擬假設**）

AI4I has no machine importance data, so impact is randomly simulated at levels 1-3 only to show the method. Mark this as simulated in the report; a real factory would use real downtime cost.
AI4I 沒有機器的重要程度，這裡隨機模擬 1–3 級，只用來展示做法。報告中要標明這是模擬，真實工廠要換成實際的停機損失。

In [ ]:
rng = np.random.default_rng(42)
impact = pd.Series(rng.choice([1, 2, 3], size=len(X_te), p=[0.5, 0.3, 0.2]), index=X_te.index)
prio = pd.DataFrame({"udi": df.loc[X_te.index, "udi"], "risk": p_cal, "impact_simulated": impact})
prio["priority"] = prio["risk"] * prio["impact_simulated"]
prio.sort_values("priority", ascending=False).head(10).round(3)

## To do / 待辦
- [ ] Compare class weight with SMOTE / 比較 class weight 與 SMOTE
- [ ] Replace cost and impact assumptions with sourced numbers / 把成本與影響的假設換成有依據的數字
- [ ] Add NASA C-MAPSS as an extension case (remaining useful life) / 加入 NASA C-MAPSS 當延伸案例
- [ ] Write calibrated predictions back to the `predictions` table / 把校正後的預測寫回 predictions 表
- [ ] Report: state limits (synthetic data, no time series, assumed costs and impact) / 報告：寫明局限性